## Read the data

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"
EXPERIMENT= "core_identification"

# 1. Read data (Now it is the same data)

In [2]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/{EXPERIMENT}/per_length_per_kfold_split/' # work laptop
PATH_TO_SAVE_RESULTS = f'{DATA_PATH_BASE}/{EXPERIMENT}/peptide_cores/'
APPROACH = "DECOY_PATHS"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
# per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
# per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'
          ]
ALLELES = [
# 'HLA-DQA1*05:01/DQB1*02:01', 
    
    'HLA-DPA1*02:02/DPB1*05:01'
    # 'HLA-DPA1*02:02/DPB1*05:01'
          ]
selected_alleles = ALLELES
ALLELES

['HLA-DPA1*02:02/DPB1*05:01']

## 2.0 Merge alleles together

In [4]:
# combination_size = len(ALLELES)
# ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
# # ALLELES = sorted([ 'HLA-A*03:01'])
# ALLELES = sorted([   'HLA-DRB1*01:01',
#                     # 'HLA-DPA1*01:03/DPB1*04:01'
#                       ])
# ALLELES = sorted(['HLA-DPA1*01:03-DPB1*04:01'])
#
# ALLELES = sorted(['HLA-DRB1*01:01', 'HLA-DPA1*01:03-DPB1*04:01'])
#

combination_size = 1
TOTAL_SPLITS = 1
TARGET_LENGTHS= [ 12, 13, 14, 15, 16, 17, 18]
# TARGET_LENGTHS= [8, 9, 10, 11, 12]
RUNS = 20
USE_WEIGHTS = True

# SELECTED_RUN = 0
# SELECTED_RUN = {0,1,2,3,4,5, 6, 7, 8, 9}
SELECTED_RUN = {0,1}

# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
models_to_read = ['v']

## Merge train data together (multiple alleles to single set of peptides)

In [5]:
from collections import defaultdict
global_selected_alleles = ALLELES # Data alleles we selected (but the number will be reduced by acceptable alleles)

selected_alleles = global_selected_alleles


per_allele_representations_train = dict()
per_allele_representations_test = dict()
per_name_representation_train = dict()
per_name_representation_test = dict()

def handle_intersections(per_allele_per_kfold_per_length_peptides):
    per_allele_result_peptide_sets =  defaultdict(set)
    per_allele_per_length_peptide_sets =  defaultdict(set)
    PER_LENGTH_SIZE = {target_length: min([len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])) for allele_name in selected_alleles]) for target_length in TARGET_LENGTHS}
    print("Min unique sizes", PER_LENGTH_SIZE)
    for target_length in TARGET_LENGTHS:
        print(f"Length {target_length}")
        for allele_name in selected_alleles:
            if target_length in per_allele_per_kfold_per_length_peptides[allele_name][0]:
                print(f"initial length for  allele { allele_name} TL {target_length} is {len(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])}")
                print(f"unique length for  allele { allele_name} TL {target_length} is {len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length]))}")
                unique_peptides = np.unique(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])
                if "imm" not in allele_name: #do not subsambple for imm
                    peptides = set(np.random.choice(unique_peptides, size=PER_LENGTH_SIZE[target_length], replace=False))
                else:
                    peptides = set(unique_peptides)
                print(f"Selected/Subsampled {len(peptides)} peptides")
                per_allele_per_length_peptide_sets[allele_name] = peptides
        # get all intersections
        intersections = list()
        for i in range(len(selected_alleles)):
            for j in range(i + 1, len(selected_alleles)):
                set1 = per_allele_per_length_peptide_sets[selected_alleles[i]]
                set2 = per_allele_per_length_peptide_sets[selected_alleles[j]]
                intersect = set1.intersection(set2)
                print(f"{selected_alleles[i]} intersect {selected_alleles[j]} - {len(intersect)}")
                intersections.append(intersect)
        # delete all intersections
        for allele_name, peptide_set in per_allele_per_length_peptide_sets.items():
            print(f"Allele {allele_name}: deleted", end=" ")
            initial_length = len(peptide_set)
            for intersect in intersections:
                peptide_set = peptide_set.difference(intersect)
            result_length = len(peptide_set)
            print(f" {initial_length - result_length} of intersected peptides")
            per_allele_result_peptide_sets[allele_name] = per_allele_result_peptide_sets[allele_name].union(peptide_set)
    return per_allele_result_peptide_sets

print("Train")
per_allele_peptide_train_sets = handle_intersections(per_allele_per_kfold_per_length_binders_train)
print("Test")
per_allele_peptide_test_sets = handle_intersections(per_allele_per_kfold_per_length_binders_test)



train_peptides_b = set(set().union(*per_allele_peptide_train_sets.values()))

test_peptides_b = set(set().union(*per_allele_peptide_test_sets.values()))

print(f"Train {len(train_peptides_b)}, Test {len(test_peptides_b)}")



data_df = pd.DataFrame(index=list(train_peptides_b))
for allele_name in selected_alleles:
    data_df.loc[list(per_allele_peptide_train_sets[allele_name]), 'allele'] = allele_name



Train
Min unique sizes {12: 420, 13: 790, 14: 1241, 15: 1231, 16: 927, 17: 628, 18: 403}
Length 12
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 12 is 420
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 12 is 420
Selected/Subsampled 420 peptides
Allele HLA-DPA1*02:02/DPB1*05:01: deleted  0 of intersected peptides
Length 13
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 13 is 790
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 13 is 790
Selected/Subsampled 790 peptides
Allele HLA-DPA1*02:02/DPB1*05:01: deleted  0 of intersected peptides
Length 14
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 14 is 1241
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 14 is 1241
Selected/Subsampled 1241 peptides
Allele HLA-DPA1*02:02/DPB1*05:01: deleted  0 of intersected peptides
Length 15
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 15 is 1231
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 15 is 1231
Selected/Subsampled 1231 peptides
Al

In [6]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets

per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

HLA-DPA1*02:02/DPB1*05:01
Length 9: HLA-DPA1*02:02/DPB1*05:01 - 12 
Length 10: HLA-DPA1*02:02/DPB1*05:01 - 53 
Length 11: HLA-DPA1*02:02/DPB1*05:01 - 204 
Length 12: HLA-DPA1*02:02/DPB1*05:01 - 420 
Length 13: HLA-DPA1*02:02/DPB1*05:01 - 790 
Length 14: HLA-DPA1*02:02/DPB1*05:01 - 1241 
Length 15: HLA-DPA1*02:02/DPB1*05:01 - 1231 
Length 16: HLA-DPA1*02:02/DPB1*05:01 - 927 
Length 17: HLA-DPA1*02:02/DPB1*05:01 - 628 
Length 18: HLA-DPA1*02:02/DPB1*05:01 - 403 
Length 19: HLA-DPA1*02:02/DPB1*05:01 - 232 
Length 20: HLA-DPA1*02:02/DPB1*05:01 - 121 
Length 21: HLA-DPA1*02:02/DPB1*05:01 - 72 
Length 22: HLA-DPA1*02:02/DPB1*05:01 - 36 
Length 23: HLA-DPA1*02:02/DPB1*05:01 - 25 
Length 24: HLA-DPA1*02:02/DPB1*05:01 - 9 
Length 25: HLA-DPA1*02:02/DPB1*05:01 - 10 
Length 26: HLA-DPA1*02:02/DPB1*05:01 - 6 


In [7]:
selected_alleles

['HLA-DPA1*02:02/DPB1*05:01']

In [8]:
data_df.allele.value_counts()

HLA-DPA1*02:02/DPB1*05:01    5640
Name: allele, dtype: int64

# Read models

In [9]:
MODELS_DESCRIPTIONS = {
    "viterbi_simple_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 7,
        "core_end_state": 8

    },
    "viterbi_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35],
        "core_end_state": [36]

    },
    "viterbi_multiple_7": {
        "path": "experiment_results/core_multiple_allele/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-DRB1*01:01',
                    'HLA-DPA1*01:03/DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35, 36],
        "core_end_state": [37, 38],
        "need_to_delete": [39, 40, 41, 42]

    },
        "viterbi_models_multiple_motifs": {
        "path": "experiment_results/core_identification/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [11,12, 13, 14, 15, 16, 17, 18, 19],
        "core_start_state": [7],
        "core_end_state": [8],
        "need_to_delete": [39, 40, 41, 42]

    },
            "viterbi_models_multiple_motifs_14": {
        "path": "experiment_results/core_identification/",
        "states": 14,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [11,12, 13, 14, 15, 16, 17, 18, 19],
        "core_start_state": [14, 15],
        "core_end_state": [16,17],
        "need_to_delete": [39, 40, 41, 42]

    },
          "viterbi_models_multiple_motifs_18x2": {
        "path": "experiment_results/core_identification/",
        "states": 18,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [11,12, 13, 14, 15, 16, 17, 18, 19],
        "core_start_state": [18,19],
        "core_end_state": [20,21],
        "need_to_delete": [39, 40, 41, 42]

    },
          "viterbi_models_multiple_motifs_35": {
        "path": "experiment_results/core_identification/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [11,12, 13, 14, 15, 16, 17, 18, 19],
        "core_start_state": [35, 36, 37],
        "core_end_state": [ 38,39, 40],
        "need_to_delete": [111]

    },
          "viterbi_models_multiple_motifs_35_12-18": {
        "path": "experiment_results/core_identification/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35, 36, 37],
        "core_end_state": [ 38,39, 40],
        "need_to_delete": [111]

    },
             "viterbi_models_multiple_motifs_7_11_19": {
        "path": "experiment_results/core_identification/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [11, 12, 13, 14, 15, 16, 17, 18, 19],
        "core_start_state": [7],
        "core_end_state": [8],
        "need_to_delete": [111]

    },
            "viterbi_models_multiple_motifs_21_11_19": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [11, 12, 13, 14, 15, 16, 17, 18, 19],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25,26],
        "need_to_delete": [111]

    },          
    "viterbi_models_multiple_motifs_21_12_18": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25,26],
        "need_to_delete": [111]

    },
    
        "viterbi_models_multiple_motifs_21_13_17": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [ 13, 14, 15, 16, 17],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25,26],
        "need_to_delete": [111]

    },
            "viterbi_models_multiple_motifs_21_12_18": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25,26],
        "need_to_delete": [111]

    },
                "viterbi_models_multiple_motifs_14_12_18": {
        "path": "experiment_results/core_identification/",
        "states": 14,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [14,15, 16],
        "core_end_state": [17,18,19],
        "need_to_delete": [111]

    },
    
          "viterbi_models_multiple_motifs_14_12_18_com1": {
        "path": "experiment_results/core_identification/",
        "states": 14,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [14],
        "core_end_state": [15],
        "need_to_delete": [111]

    },
             "viterbi_models_multiple_motifs_21_12_18_comp3": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [ 12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25,26],
        "need_to_delete": [111]

    },
    
                 "viterbi_models_multiple_motifs_21_12_18_comp2": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [ 12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21,22,],
        "core_end_state": [23, 23],
        "need_to_delete": [111]

    },
       "viterbi_models_multiple_motifs_21_12_18_com3": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25, 26],
        "need_to_delete": [111]

    },
                   "viterbi_models_multiple_motifs_14_12_18_com3": {
        "path": "experiment_results/core_identification/",
        "states": 14,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [14,15, 16],
        "core_end_state": [17,18, 19],
        "need_to_delete": [111]

    },
                   "core_identification_pseudocount_anchor": {
        "path": "experiment_results/core_identification_pseudocount_anchor/",
        "states": 14,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 
                'HLA-DPA1*02:02/DPB1*05:01'

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [14,15, 16],
        "core_end_state": [17,18, 19],
        "need_to_delete": [111]

    },














    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [10]:
selected_models = [
     "core_identification_pseudocount_anchor"
]


In [11]:
from hmm_logic_methods import load_model

In [12]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_binding_model = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_binding_model[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [13]:
model_alleles = model_description['alleles']
MIXED_ALLELES = list()

model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
for combination in itertools.combinations(sorted(model_alleles), combination_size):
    allele_name = "+".join(combination)
    # allele_name = f"{allele_name1}+{allele_name2}"
    MIXED_ALLELES.append(allele_name)

In [14]:
model_description['alleles']

['HLA-DQA1*05:01/DQB1*02:01', 'HLA-DPA1*02:02/DPB1*05:01']

In [15]:
OLD_ALLELES

['HLA-DPA1*02:02/DPB1*05:01']

In [16]:
model_alleles

{'HLA-DPA1*02:02/DPB1*05:01'}

In [17]:
MODELS_DESCRIPTIONS[model]

{'path': 'experiment_results/core_identification_pseudocount_anchor/',
 'states': 14,
 'params_string': '30-min_iters',
 'combination_size': 1,
 'alleles': ['HLA-DQA1*05:01/DQB1*02:01', 'HLA-DPA1*02:02/DPB1*05:01'],
 'target_lengths': [12, 13, 14, 15, 16, 17, 18],
 'core_start_state': [14, 15, 16],
 'core_end_state': [17, 18, 19],
 'need_to_delete': [111]}

In [18]:
MODELS_DESCRIPTIONS[model]

{'path': 'experiment_results/core_identification_pseudocount_anchor/',
 'states': 14,
 'params_string': '30-min_iters',
 'combination_size': 1,
 'alleles': ['HLA-DQA1*05:01/DQB1*02:01', 'HLA-DPA1*02:02/DPB1*05:01'],
 'target_lengths': [12, 13, 14, 15, 16, 17, 18],
 'core_start_state': [14, 15, 16],
 'core_end_state': [17, 18, 19],
 'need_to_delete': [111]}

## Mark important states in model(will be used for core call)

In [19]:
import logomaker
from collections import defaultdict
def probs_to_info_matrix(distribution):
    amino_acids = list(distribution.parameters[0].keys())
    dummy_df = pd.DataFrame(columns = sorted(amino_acids), index=[0],  dtype=float)
    for i, (acid, prob) in enumerate(distribution.parameters[0].items()):
        dummy_df.loc[0, acid] = prob
    dummy_df.index.name = "pos"
    info_matrix = logomaker.transform_matrix(dummy_df, from_type='probability', to_type='information')
    return info_matrix

per_model_total_df = dict()

for model_key, model in per_name_binding_model.items():
    per_state_information = dict()
    per_state_distribution = dict()
    per_level_information = defaultdict(float)

    for state in model.states:
        if state not in [model.start, model.end]:
            info_matrix = probs_to_info_matrix(state.distribution)
            per_state_distribution[state.name] = state.distribution.parameters[0]
            per_state_information[state.name] = sum(info_matrix.loc[0])
            # per_level_information[name_to_level[state.name]] += per_state_information[state.name]
    info_df = pd.DataFrame(per_state_information.items(), columns=["state", "info"])
    distr_df = pd.DataFrame(per_state_distribution.values())
    total_df = pd.concat([info_df, distr_df], axis=1)
    per_model_total_df[model_key] = total_df

In [20]:
filtered_dfs = list()
per_model_important_states = dict()
if models_to_read == ['bw']:
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df[total_df['info'] >= 1]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)

if models_to_read == ['v']:
    per_model_anchors = dict()
    per_name_models = per_name_binding_model

    core_start_states= MODELS_DESCRIPTIONS[selected_models[0]]['core_start_state']
    core_end_states = MODELS_DESCRIPTIONS[selected_models[0]]['core_end_state']

    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[core_start_states + core_end_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)





## Calculate core guesses

In [21]:
# prepare peptides for viterbi calculation (expected to be unique)
allele_peptides = data_df.index.values


len(allele_peptides)

5640

In [22]:
TEST_PDB_CORES = False
if TEST_PDB_CORES:
    df=pd.read_csv('test/binding_cores', sep='\t')
    df= df[~df['Antigen'].str.contains('X')]
    df = df[df['Allele'] == 'DRB1*01:01']

    allele_peptides = df.Antigen.values
    true_labels=df['Core (PDB)'].values
    netmhcpan = df['Predicted core.5'].values

In [23]:
from hmm_logic_methods import calculate_model_core_guesses
selected_peptides = [
    # "APEEILAEKSFNTVPY", 'EALTFSREHGPGVAEPE',
    #                  'YYSLLMPILTLTRAL',
    #                      'SYLTNMFATWSP',
    #                  'NPLIRHENRMVLAST',
    #                      'KPPKPVSKMRMATPLLM',
                         ]
total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states,
                                    selected_peptides
                                   )

Model 0_run-12-13-14-15-16-17-18-14-st-viterbi-alg-30-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████████████████████████████████████████████████████████████████████████████| 5640/5640 [00:00<00:00, 40048.61it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 5640 


100%|██████████████████████████████████████████████████████████████████████████████████| 5640/5640 [00:00<00:00, 25560.13it/s]


Model 1_run-12-13-14-15-16-17-18-14-st-viterbi-alg-30-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████████████████████████████████████████████████████████████████████████████| 5640/5640 [00:00<00:00, 36958.90it/s]


... (70 outputs left out)

Model 19_run-12-13-14-15-16-17-18-14-st-viterbi-alg-30-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████████████████████████████████████████████████████████████████████████████| 5640/5640 [00:00<00:00, 36635.62it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 5640 


100%|██████████████████████████████████████████████████████████████████████████████████| 5640/5640 [00:00<00:00, 24536.49it/s]


In [24]:
data_df = data_df.join(total_df)


In [25]:
data_df.head()

,allele,simplified_path,core_start_0,core_end_0,prob_0,core_start_1,core_end_1,prob_1,core_start_2,core_end_2,...,core_start_17,core_end_17,prob_17,core_start_18,core_end_18,prob_18,core_start_19,core_end_19,prob_19,peptide
VKYLATLETLTKH,HLA-DPA1*02:02/DPB1*05:01,"[1.06, 8.44, 10.650000000000002, 20.9999999999...",0,9,-2.895222,1,11,-2.635957,1,11,...,3,11,-2.689272,1,10,-2.739722,2,11,-3.020556,VKYLATLETLTKH
RRNSVFQQGMKNKIL,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 5.48, 16.400000000000002, 18.64, 20.7800...",2,11,-3.214922,1,10,-3.283023,4,13,...,2,10,-3.156869,1,10,-3.087138,3,12,-3.179123,RRNSVFQQGMKNKIL
GQPGKVIEVKAKRRT,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 0.0, 6.33, 13.72, 20.08, 21.129999999999...",3,12,-2.958426,4,13,-2.862347,5,13,...,2,11,-2.798302,2,11,-2.763822,4,12,-2.862809,GQPGKVIEVKAKRRT
RPTSLPLNTKNSTKEPR,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 10.56, 10.56, 13.780000000000003, 15.97,...",1,11,-2.968839,6,16,-3.081609,6,15,...,1,9,-2.997962,3,12,-2.991424,3,11,-3.000289,RPTSLPLNTKNSTKEPR
DVMTTVFKVGAK,HLA-DPA1*02:02/DPB1*05:01,"[2.14, 17.300000000000004, 20.560000000000002,...",0,8,-3.087383,1,10,-2.727074,1,10,...,1,9,-3.242689,1,10,-3.040687,1,10,-3.082133,DVMTTVFKVGAK


## Set core based on most popular value

In [26]:
models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]

In [27]:
core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
core_ends = models_predictions_end.mode(axis=1)[0].astype(int)

In [28]:
final_df = total_df

In [29]:
selected_cluster_column = 'min_entropy_cluster'
#selected_cluster_column = 'max_mean_prob_cluster'
# final_df['core_start'] = 0
# final_df['core_end'] = 0
# for cluster in cluster_descriptions.index:
#     index_peptides = final_df[final_df[selected_cluster_column] == cluster].index
#     start_values = cluster_to_start_columns[cluster]
#     end_values = cluster_to_end_columns[cluster]
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mode(axis=1)[0].astype(int)

#     #final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mode(axis=1)[0].astype(int)

final_df['core_start'] = core_starts
final_df['core_end'] = core_ends

final_df['core_peptide'] = final_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
final_df['core_len'] = final_df['core_peptide'].str.len()

In [30]:
final_df['core_len'].value_counts()

9     4317
10     885
7      141
11     114
8       87
12      69
13      10
15       6
5        5
14       3
6        3
Name: core_len, dtype: int64

In [31]:
data_df['core_start'] = final_df['core_start']
data_df['core_end'] = final_df['core_end']
data_df['core_len'] = final_df['core_len']
data_df['peptide_len'] = data_df.index.str.len()
data_df['core_peptide'] = final_df['core_peptide']
data_df = data_df[~data_df.index.duplicated(keep='first')]

In [32]:
data_df['core_start'].value_counts()

1    1686
2    1637
3    1151
4     569
0     268
5     193
6      91
7      41
8       4
Name: core_start, dtype: int64

# Save core data to files

In [33]:
import os
if not os.path.exists(PATH_TO_SAVE_RESULTS):
    os.makedirs(PATH_TO_SAVE_RESULTS)
data_df.to_csv("{}extractd_cores.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=True)

In [34]:
data_df

,allele,simplified_path,core_start_0,core_end_0,prob_0,core_start_1,core_end_1,prob_1,core_start_2,core_end_2,...,prob_23,core_start_24,core_end_24,prob_24,peptide,core_start,core_end,core_len,peptide_len,core_peptide
IVTKDDAMLLKGKGDK,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 1.07, 7.820000000000001, 25.6, 25.6, 26....",3,12,-2.893736,3,11,-2.954684,3,11,...,-2.960458,6,14,-2.998371,IVTKDDAMLLKGKGDK,3,11,9,16,KDDAMLLKG
AVRALKSLSKKYR,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 2.08, 11.54, 26.389999999999997, 26.3899...",3,11,-2.775002,2,10,-2.670610,2,10,...,-2.635512,3,11,-2.816089,AVRALKSLSKKYR,3,11,9,13,ALKSLSKKY
SEYLRALFVYEKG,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 1.13, 7.790000000000001, 19.120000000000...",2,11,-2.973951,4,12,-2.993019,4,12,...,-3.155080,3,11,-3.166252,SEYLRALFVYEKG,3,11,9,13,LRALFVYEK
IVEKIGEEFLTDLSQ,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 1.06, 7.57, 19.389999999999997, 25.98, 2...",3,12,-2.906430,3,11,-2.940935,3,11,...,-3.033744,3,12,-2.924462,IVEKIGEEFLTDLSQ,3,12,10,15,KIGEEFLTDL
LPTDVMTTVFKVGAKTD,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 0.0, 2.1500000000000004, 10.590000000000...",4,13,-2.940326,3,11,-2.943551,3,11,...,-2.994453,4,12,-2.875853,LPTDVMTTVFKVGAKTD,4,12,9,17,VMTTVFKVG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LPGHVQAYQEAVKSG,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 0.0, 0.0, 1.11, 3.1900000000000004, 12.5...",6,14,-3.177504,5,13,-2.829845,5,13,...,-3.116389,6,14,-2.956966,LPGHVQAYQEAVKSG,6,14,9,15,AYQEAVKSG
DHNFVKAINAIQKSWTA,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 0.0, 0.0, 0.0, 2.1, 21.520000000000003, ...",6,15,-3.110044,5,13,-3.061790,5,13,...,-3.214073,6,14,-3.292185,DHNFVKAINAIQKSWTA,5,13,9,17,KAINAIQKS
ALEVMESVVKNCGQT,HLA-DPA1*02:02/DPB1*05:01,"[0.0, 2.0700000000000003, 11.389999999999999, ...",3,12,-2.866342,2,10,-2.818211,2,10,...,-2.989331,3,11,-2.900333,ALEVMESVVKNCGQT,3,11,9,15,VMESVVKNC
IERIPELNKVAR,HLA-DPA1*02:02/DPB1*05:01,"[2.1500000000000004, 11.760000000000002, 26.77...",2,10,-2.868351,1,9,-2.801553,1,9,...,-2.842928,2,10,-2.746109,IERIPELNKVAR,2,10,9,12,RIPELNKVA


# Group models based on predictions

We got all the start predictions from all models. Now we want to group these models. (since we expect that models have disagrement)

We want to cluster columns based on how similar are predictions for the whole training data



In [65]:
models_predictions = total_df[[f"core_start_{i}" for i in range(RUNS)]]

In [ ]:
from sklearn.cluster import FeatureAgglomeration
distance_threshold = 150

agglo = FeatureAgglomeration(n_clusters=None, metric='euclidean', linkage='ward', pooling_func=np.mean, distance_threshold=distance_threshold)
result_start_df = agglo.fit_transform(models_predictions)

In [78]:
clustered_features = pd.DataFrame({"cluster": agglo.labels_})

clustered_features['model'] = models_predictions.columns


unique_labels_counts = clustered_features.cluster.value_counts().sort_values(ascending=False)
unique_labels_counts

1    15
2    10
3     6
0     4
Name: cluster, dtype: int64

In [79]:
cluster_descriptions = clustered_features.groupby('cluster').agg(lambda x: " ".join(x.values))
cluster_descriptions['len'] = unique_labels_counts
cluster_descriptions.sort_values(by='len', ascending=False)

,model,len
cluster,,
1,core_start_1 core_start_3 core_start_10 core_s...,15
2,core_start_2 core_start_5 core_start_6 core_st...,10
3,core_start_9 core_start_11 core_start_23 core_...,6
0,core_start_0 core_start_4 core_start_17 core_s...,4


In [80]:
cluster_to_start_columns = {i : total_df[cluster_descriptions.loc[i]['model'].split(' ')] for i in cluster_descriptions.index}

In [81]:
model_to_cluster = dict(clustered_features['cluster'])

## Add mean probability for cluster for each peptide

In [34]:
from collections import defaultdict
cluster_to_prob_column_names = defaultdict(list)
cluster_to_end_column_names = defaultdict(list)

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = 0
    
# calculate mean for predictions for given cluster of models to 
for i in range(RUNS):
    total_df[f'cluster_prob_{model_to_cluster[i]}'] += total_df[f'prob_{i}']
    cluster_to_prob_column_names[model_to_cluster[i]].append(f'prob_{i}')
    cluster_to_end_column_names[model_to_cluster[i]].append(f'core_end_{i}')

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = total_df[f'cluster_prob_{cluster}'] / cluster_descriptions.loc[cluster]['len']
    
total_df['max_mean_prob_cluster'] =   total_df[[f'cluster_prob_{cluster}' for cluster in cluster_descriptions.index]].idxmax(axis=1).str.split('_').map(lambda x : int(x[-1]))

cluster_to_prob_columns = {i : total_df[cluster_to_prob_column_names[i]] for i in cluster_to_prob_column_names}
cluster_to_end_columns = {i : total_df[cluster_to_end_column_names[i]] for i in cluster_to_end_column_names}


0
1
2
3
0
1
2
3


In [35]:
cluster_to_start_columns[1].shape

(5640, 15)

## Add entropy value for cluster for each column

In [36]:
def mean_predictive_entropy(predictions):
    entropy = np.mean(-np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1])
    return entropy

def predictive_entropy(predictions):
    entropy = -np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1]
    return entropy

print(mean_predictive_entropy(cluster_to_prob_columns[0]))
print(mean_predictive_entropy(cluster_to_prob_columns[1]))
#print(mean_predictive_entropy(cluster_to_prob_columns[2]))
#print(mean_predictive_entropy(cluster_to_prob_columns[3]))

cluster_to_start_entropy_df = {i: predictive_entropy(cluster_to_prob_columns[i]) for i in cluster_to_prob_columns.keys()}

0.15289490292766392
0.15267749732179514


In [37]:
df_concat = pd.concat(cluster_to_start_entropy_df.values(), axis=1)
min_entropy_clusters = df_concat.idxmin(axis=1)
df_concat.columns = [f'cluster_entropy_{cluster}' for cluster in df_concat.columns ]
df_concat['min_entropy_cluster'] = min_entropy_clusters

In [38]:
final_df = pd.concat([total_df, df_concat], axis=1)